In [1]:
import os
import pandas as pd
import joblib
from collections import defaultdict

# ===============================
# PATHS
# ===============================
coloc_path    = "/n/scratch/users/a/adm808/Revision/Coloc/coloc_results_AD_GWAS_GTEx_v10_Cortex_and_BA9_all_brain.csv"
seaad_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"
seaad_de_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed"

seaad_de_files = {
    "Ast": "poisson_DE_results_SEAAD_Ast_COMBINED.csv",
    "Mic": "poisson_DE_results_SEAAD_Mic_COMBINED.csv",
    "Inh": "poisson_DE_results_SEAAD_Inh_COMBINED.csv",
    "Oli": "poisson_DE_results_SEAAD_Oli_COMBINED.csv",
    "Opc": "poisson_DE_results_SEAAD_Opc_COMBINED.csv",
    "Ex":  "poisson_DE_results_SEAAD_Ex_COMBINED.csv",
}

# Canonical (SEAAD-style) cell-type labels for iteration.
canonical_cell_types = ["Ast", "Mic", "Inh", "Oli", "Opc", "Ex"]

# Coloc CSV uses 'In' (ROSMAP-style). Map canonical -> coloc label.
coloc_ct_map = {"Ast": "Ast", "Mic": "Mic", "Inh": "In", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}

# Thresholds
PPH4_THRESHOLD     = 0.8
ML_MIN_SPLITS      = 2
DEG_P_ADJ_CUTOFF   = 0.05
DEG_ABS_LOG2FC     = 0.25

# ===============================
# 1. COLOC-SUPPORTED GENES PER CELL TYPE (PP.H4 > 0.8, max across tissues)
# ===============================
coloc_df = pd.read_csv(coloc_path)
required = {"gene_symbol", "cell_type", "PP.H4.abf"}
missing = required - set(coloc_df.columns)
if missing:
    raise ValueError(f"coloc CSV missing columns: {missing}")

# Take max PP.H4 per gene × coloc cell_type
agg = (
    coloc_df
    .groupby(["gene_symbol", "cell_type"])["PP.H4.abf"]
    .max()
    .reset_index()
)

coloc_genes = {}
for ct in canonical_cell_types:
    coloc_label = coloc_ct_map[ct]
    sub = agg[(agg["cell_type"] == coloc_label) & (agg["PP.H4.abf"] > PPH4_THRESHOLD)]
    coloc_genes[ct] = set(sub["gene_symbol"].astype(str))

# ===============================
# 2. SEAAD ML PREDICTORS (>= 2/5 splits)
# ===============================
ml_genes = defaultdict(set)
for ct in canonical_cell_types:
    gene_counts = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(seaad_ml_base, ct, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1
    ml_genes[ct] = {g for g, c in gene_counts.items() if c >= ML_MIN_SPLITS}

# ===============================
# 3. SEAAD DEGs (to EXCLUDE)
# ===============================
deg_genes = {}
for ct in canonical_cell_types:
    df = pd.read_csv(os.path.join(seaad_de_base, seaad_de_files[ct]))
    df = df[(df["p_adj"] < DEG_P_ADJ_CUTOFF) & (df["log2FC"].abs() > DEG_ABS_LOG2FC)]
    deg_genes[ct] = set(df["gene"].astype(str))

# ===============================
# 4. Coloc-supported ∩ ML-predictive  MINUS  DEG  — per cell type
# ===============================
print(f"=== Genes with coloc support (PP.H4 > {PPH4_THRESHOLD}) AND ML-predictive (>= {ML_MIN_SPLITS}/5 splits)")
print(f"    BUT NOT a SEAAD DEG (p_adj<{DEG_P_ADJ_CUTOFF}, |log2FC|>{DEG_ABS_LOG2FC}) ===\n")

print(f"{'cell_type':<6} {'coloc':>7} {'ML':>7} {'DEG':>7} {'coloc∩ML':>10} {'minus DEG':>11}")
for ct in canonical_cell_types:
    c, m, d = coloc_genes[ct], ml_genes[ct], deg_genes[ct]
    coloc_and_ml = c & m
    final = coloc_and_ml - d
    print(f"{ct:<6} {len(c):>7} {len(m):>7} {len(d):>7} {len(coloc_and_ml):>10} {len(final):>11}")

print("\n=== Genes per cell type ===")
for ct in canonical_cell_types:
    final = (coloc_genes[ct] & ml_genes[ct]) - deg_genes[ct]
    print(f"\n{ct} ({len(final)} genes):")
    print(sorted(final))


=== Genes with coloc support (PP.H4 > 0.8) AND ML-predictive (>= 2/5 splits)
    BUT NOT a SEAAD DEG (p_adj<0.05, |log2FC|>0.25) ===

cell_type   coloc      ML     DEG   coloc∩ML   minus DEG
Ast          2      40     803          1           1
Mic          9     559     398          2           2
Inh          2      24     566          1           1
Oli          6      25     392          1           1
Opc          8      29     199          0           0
Ex           3      16    1360          1           0

=== Genes per cell type ===

Ast (1 genes):
['ARL17B']

Mic (2 genes):
['ARL17B', 'JAZF1']

Inh (1 genes):
['ARL17B']

Oli (1 genes):
['ARL17B']

Opc (0 genes):
[]

Ex (0 genes):
[]


In [2]:
import os
import pandas as pd
import joblib
from collections import defaultdict

# ===============================
# PATHS — ROSMAP
# ===============================
coloc_path     = "/n/scratch/users/a/adm808/Revision/Coloc/coloc_results_AD_GWAS_GTEx_v10_Cortex_and_BA9_all_brain.csv"
rosmap_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
rosmap_de_base = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"

rosmap_de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In":  "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex":  "poisson_DE_results_Ex.csv",
}

# ROSMAP uses 'In' everywhere (ML folders, DEG files, coloc CSV) — no mapping needed.
cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# Thresholds
PPH4_THRESHOLD   = 0.8
ML_MIN_SPLITS    = 2
DEG_P_ADJ_CUTOFF = 0.05
DEG_ABS_LOG2FC   = 0.25

# ===============================
# 1. COLOC-SUPPORTED GENES PER CELL TYPE (PP.H4 > 0.8, max across tissues)
# ===============================
coloc_df = pd.read_csv(coloc_path)
required = {"gene_symbol", "cell_type", "PP.H4.abf"}
missing = required - set(coloc_df.columns)
if missing:
    raise ValueError(f"coloc CSV missing columns: {missing}")

agg = (
    coloc_df
    .groupby(["gene_symbol", "cell_type"])["PP.H4.abf"]
    .max()
    .reset_index()
)

coloc_genes = {}
for ct in cell_types:
    sub = agg[(agg["cell_type"] == ct) & (agg["PP.H4.abf"] > PPH4_THRESHOLD)]
    coloc_genes[ct] = set(sub["gene_symbol"].astype(str))

# ===============================
# 2. ROSMAP ML PREDICTORS (>= 2/5 splits)
# ===============================
ml_genes = defaultdict(set)
for ct in cell_types:
    gene_counts = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(rosmap_ml_base, ct, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1
    ml_genes[ct] = {g for g, c in gene_counts.items() if c >= ML_MIN_SPLITS}

# ===============================
# 3. ROSMAP DEGs (to EXCLUDE)
# ===============================
deg_genes = {}
for ct in cell_types:
    df = pd.read_csv(os.path.join(rosmap_de_base, rosmap_de_files[ct]))
    df = df[(df["p_adj"] < DEG_P_ADJ_CUTOFF) & (df["log2FC"].abs() > DEG_ABS_LOG2FC)]
    deg_genes[ct] = set(df["gene"].astype(str))

# ===============================
# 4. Coloc-supported ∩ ML-predictive  MINUS  DEG  — per cell type
# ===============================
print(f"=== Genes with coloc support (PP.H4 > {PPH4_THRESHOLD}) AND ML-predictive (>= {ML_MIN_SPLITS}/5 splits)")
print(f"    BUT NOT a ROSMAP DEG (p_adj<{DEG_P_ADJ_CUTOFF}, |log2FC|>{DEG_ABS_LOG2FC}) ===\n")

print(f"{'cell_type':<6} {'coloc':>7} {'ML':>7} {'DEG':>7} {'coloc∩ML':>10} {'minus DEG':>11}")
for ct in cell_types:
    c, m, d = coloc_genes[ct], ml_genes[ct], deg_genes[ct]
    coloc_and_ml = c & m
    final = coloc_and_ml - d
    print(f"{ct:<6} {len(c):>7} {len(m):>7} {len(d):>7} {len(coloc_and_ml):>10} {len(final):>11}")

print("\n=== Genes per cell type ===")
for ct in cell_types:
    final = (coloc_genes[ct] & ml_genes[ct]) - deg_genes[ct]
    print(f"\n{ct} ({len(final)} genes):")
    print(sorted(final))


=== Genes with coloc support (PP.H4 > 0.8) AND ML-predictive (>= 2/5 splits)
    BUT NOT a ROSMAP DEG (p_adj<0.05, |log2FC|>0.25) ===

cell_type   coloc      ML     DEG   coloc∩ML   minus DEG
Ast          2     175     178          2           2
Mic          9     466      88          9           8
In           2     108     240          2           2
Oli          6     621     711          6           5
Opc          8     835      64          8           7
Ex           3     162    1066          3           3

=== Genes per cell type ===

Ast (2 genes):
['ARL17B', 'KNOP1']

Mic (8 genes):
['ARL17B', 'ERC2', 'JAZF1', 'RIN3', 'SYK', 'TMEM163', 'UBASH3B', 'USP6NL']

In (2 genes):
['ARL17B', 'LRRFIP2']

Oli (5 genes):
['ARL17B', 'CLU', 'KNOP1', 'NDUFAF6', 'PLEKHA1']

Opc (7 genes):
['EGFR', 'ERC2', 'HP1BP3', 'JAZF1', 'SLTM', 'TMEM163', 'ZKSCAN1']

Ex (3 genes):
['ARL17B', 'CLU', 'ERC2']


In [3]:
import os
import joblib
from collections import defaultdict

# ===============================
# SEAAD ML PREDICTORS (>= 2/5 splits)  — for cross-dataset overlap check
# ===============================
seaad_ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_seaad"

# ROSMAP cell label -> SEAAD cell label (folder name in seaad_ml_base)
rosmap_to_seaad_ct = {"Ast": "Ast", "Mic": "Mic", "In": "Inh", "Oli": "Oli", "Opc": "Opc", "Ex": "Ex"}

SEAAD_ML_MIN_SPLITS = 2

seaad_ml_genes = {}
for ct in cell_types:
    seaad_ct = rosmap_to_seaad_ct[ct]
    gene_counts = defaultdict(int)
    for split in range(1, 6):
        path = os.path.join(seaad_ml_base, seaad_ct, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue
        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1
    seaad_ml_genes[ct] = {g for g, c in gene_counts.items() if c >= SEAAD_ML_MIN_SPLITS}

# ===============================
# Cross-dataset check:
# Of the ROSMAP "coloc ∩ ML, minus DEG" genes, how many are ALSO ML-predictive in SEAAD?
# ===============================
print(f"=== ROSMAP coloc∩ML\\DEG  --> also ML-predictive in SEAAD (>= {SEAAD_ML_MIN_SPLITS}/5 splits) ===\n")
print(f"{'cell_type':<6} {'ROSMAP set':>11} {'SEAAD ML':>10} {'overlap':>9}")

for ct in cell_types:
    rosmap_set = (coloc_genes[ct] & ml_genes[ct]) - deg_genes[ct]
    seaad_set  = seaad_ml_genes[ct]
    overlap    = rosmap_set & seaad_set
    print(f"{ct:<6} {len(rosmap_set):>11} {len(seaad_set):>10} {len(overlap):>9}")

print("\n=== Overlapping genes per cell type ===")
for ct in cell_types:
    rosmap_set = (coloc_genes[ct] & ml_genes[ct]) - deg_genes[ct]
    overlap    = rosmap_set & seaad_ml_genes[ct]
    print(f"\n{ct} ({len(overlap)} of {len(rosmap_set)} ROSMAP-set genes are SEAAD-predictive):")
    print(sorted(overlap))


=== ROSMAP coloc∩ML\DEG  --> also ML-predictive in SEAAD (>= 2/5 splits) ===

cell_type  ROSMAP set   SEAAD ML   overlap
Ast              2         40         1
Mic              8        559         2
In               2         24         1
Oli              5         25         1
Opc              7         29         0
Ex               3         16         1

=== Overlapping genes per cell type ===

Ast (1 of 2 ROSMAP-set genes are SEAAD-predictive):
['ARL17B']

Mic (2 of 8 ROSMAP-set genes are SEAAD-predictive):
['ARL17B', 'JAZF1']

In (1 of 2 ROSMAP-set genes are SEAAD-predictive):
['ARL17B']

Oli (1 of 5 ROSMAP-set genes are SEAAD-predictive):
['ARL17B']

Opc (0 of 7 ROSMAP-set genes are SEAAD-predictive):
[]

Ex (1 of 3 ROSMAP-set genes are SEAAD-predictive):
['ARL17B']
